# Mono3D Cloud Export for vehicle_demo\n\nUpload `mono3d_target7_cloud_input.zip`, run Mono3D on Linux/GPU, then download `mono3d_target7_outputs.json` and compare it locally with `compare_mono3d_centers.py`.\n

## 1. Upload package\nRun the local packaging script first, then upload the generated zip here.\n

In [ ]:
from google.colab import files\nuploaded = files.upload()\nzip_path = next(iter(uploaded.keys()))\nprint('uploaded:', zip_path)\n

In [ ]:
import json, os, zipfile, pathlib, shutil\nWORK = pathlib.Path('/content/mono3d_target7')\nif WORK.exists(): shutil.rmtree(WORK)\nWORK.mkdir(parents=True)\nwith zipfile.ZipFile(zip_path, 'r') as zf:\n    zf.extractall(WORK)\nmanifest = json.load(open(WORK / 'manifest.json'))\nprint('frames:', len(manifest['frames']))\nprint('first:', manifest['frames'][0])\n

## 2. Install MMDetection3D\nThis follows the official OpenMMLab/MMDetection3D install style. If Colab changes CUDA/Torch versions, use the current MMDetection3D install guide to adjust the mmcv wheel command.\n

In [ ]:
!python -m pip install -U pip setuptools wheel openmim\n!mim install 'mmengine>=0.7.1'\n!mim install 'mmcv>=2.0.0,<2.2.0'\n!mim install 'mmdet>=3.0.0,<3.4.0'\n!python -m pip install 'mmdet3d==1.4.0'\n

## 3. Download a Mono3D model\nStart with SMOKE because it directly predicts monocular 3D boxes. If the config name changes, run `!mim search mmdet3d --model smoke` and replace `CONFIG_NAME`.\n

In [ ]:
CONFIG_NAME = 'smoke_dla34_dlaneck_gn-all_4xb8-6x_kitti-mono3d'\n!mkdir -p /content/mmdet3d_checkpoints\n!mim download mmdet3d --config $CONFIG_NAME --dest /content/mmdet3d_checkpoints\n!ls -lh /content/mmdet3d_checkpoints\n

## 4. Run inference and export normalized JSON\nThe exact MMDetection3D mono inference API may differ by version. The cell below tries the standard API first. If your installed version exposes a different inferencer, adjust only this cell; the output schema should stay unchanged.\n

In [ ]:
import glob, json, pathlib, numpy as np\nfrom PIL import Image\nfrom mmengine.config import Config\nfrom mmdet3d.apis import init_model\ntry:\n    from mmdet3d.apis import inference_mono_3d_detector\nexcept Exception as e:\n    inference_mono_3d_detector = None\n    print('inference_mono_3d_detector unavailable:', e)\n\ncfg_path = glob.glob('/content/mmdet3d_checkpoints/*.py')[0]\nckpt_path = glob.glob('/content/mmdet3d_checkpoints/*.pth')[0]\nprint(cfg_path, ckpt_path)\nmodel = init_model(cfg_path, ckpt_path, device='cuda:0')\n\ndef make_ann(frame, ann_path):\n    cam2img = frame.get('cam2img')\n    if cam2img is None:\n        # Fallback approximate intrinsics; replace with calibrated K if needed.\n        w, h = frame['width'], frame['height']\n        cam2img = [[w, 0, w / 2], [0, w, h / 2], [0, 0, 1]]\n    ann = {\n        'images': [{'id': 0, 'file_name': frame['cloud_image_path'], 'width': frame['width'], 'height': frame['height'], 'cam2img': cam2img}],\n        'annotations': [],\n        'categories': [{'id': 0, 'name': 'car'}]\n    }\n    json.dump(ann, open(ann_path, 'w'))\n    return ann_path\n\ndef tensor_to_list(x):\n    if hasattr(x, 'detach'):\n        x = x.detach().cpu().numpy()\n    if hasattr(x, 'tolist'):\n        return x.tolist()\n    return x\n\nout_frames = []\nfor frame in manifest['frames']:\n    image_path = str(WORK / frame['cloud_image_path'])\n    ann_path = str(WORK / f"ann_{frame['frame_index']:06d}.json")\n    make_ann(frame, ann_path)\n    detections = []\n    if inference_mono_3d_detector is None:\n        raise RuntimeError('Please adjust this cell for your mmdet3d version/inferencer.')\n    result = inference_mono_3d_detector(model, image_path, ann_path)\n    # MMDetection3D versions differ. Inspect result once if this parser fails.\n    pred = result.pred_instances_3d if hasattr(result, 'pred_instances_3d') else result[0].pred_instances_3d\n    scores = tensor_to_list(getattr(pred, 'scores_3d', []))\n    labels = tensor_to_list(getattr(pred, 'labels_3d', []))\n    bboxes_3d = getattr(pred, 'bboxes_3d', None)\n    centers = []\n    if bboxes_3d is not None:\n        centers = tensor_to_list(bboxes_3d.gravity_center) if hasattr(bboxes_3d, 'gravity_center') else tensor_to_list(bboxes_3d.tensor[:, :3])\n    pred2d = result.pred_instances if hasattr(result, 'pred_instances') else getattr(result[0], 'pred_instances', None)\n    boxes2d = tensor_to_list(getattr(pred2d, 'bboxes', [])) if pred2d is not None else []\n    for i, c in enumerate(centers):\n        detections.append({\n            'box_xyxy': boxes2d[i] if i < len(boxes2d) else None,\n            'mono3d_center_m': [float(v) for v in c],\n            'mono3d_score': float(scores[i]) if i < len(scores) else None,\n            'mono3d_label_id': int(labels[i]) if i < len(labels) else None,\n            'label': 'car'\n        })\n    out_frames.append({'frame_index': frame['frame_index'], 'image': frame['original_path'], 'detections': detections})\n\nout = {'meta': {'backend': 'mmdet3d_smoke', 'config': CONFIG_NAME}, 'frames': out_frames}\nout_path = WORK / 'mono3d_target7_outputs.json'\njson.dump(out, open(out_path, 'w'), ensure_ascii=False, indent=2)\nprint('wrote', out_path)\n

In [ ]:
from google.colab import files\nfiles.download(str(WORK / 'mono3d_target7_outputs.json'))\n